<a href="https://colab.research.google.com/github/Jef-H/super_classy_fields/blob/develop/super_classy_fields_FP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Learning Classification: Field or Not a Field?

## Project Overview

This project aims to train a model to classify satellite images into two categories:
- **Field** (most likely corn, soy, or alfalfa) based on the field dataset being chosen from the great state of Iowa.
- **Not a Field**

## Data Sources

### Field Images
- Screenshots of cornfields were taken and cropped to match the Kaggle dataset size (64x64 pixels).
- Data obtained from **ESRI layers at USGS** via [Earth Explorer](https://earthexplorer.usgs.gov/).

### Other Category Images
- Non-field images were sourced from a Kaggle dataset:  
  [Satellite Image Classification Dataset](https://www.kaggle.com/datasets/mahmoudreda55/satellite-image-classification/data).  
  Categories include:
  - Cloudy
  - Desert
  - Green
  - Water

## Goals
The objective is to see how well the model can distinguish between "Field" and "Not a Field" categories using these datasets.

---

Let's see how it goes!



## imports

In [1]:
# General purpose libraries
import os
import time
import shutil
import zipfile
import tempfile
import requests

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
from PIL import Image

# Machine learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Flatten,
    Dense,
    Dropout,
    Input
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
def download_png_files_from_github(github_url, output_dir):
    """
    Downloads all PNG files from a specified GitHub repository directory,
    clearing the destination directory before downloading.

    Args:
        github_url (str): The GitHub URL of the repository directory.
        output_dir (str): Directory to save the downloaded PNG files.
    """
    # Ensure the output directory exists and clean it
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove all files in the directory
    os.makedirs(output_dir, exist_ok=True)
    print(f"Cleaned and prepared output directory: {output_dir}")

    # Extract the repository name and directory from the GitHub URL
    repo_url = github_url.split("https://github.com/")[1]
    repo_name, tree_path = repo_url.split("/tree/")

    # Correctly format the path for the API (removing 'tree' and using branch name)
    branch_name = tree_path.split('/')[0]  # Assuming branch name is the first part of the path
    dir_path = '/'.join(tree_path.split('/')[1:])  # Directory path is the rest

    # GitHub API URL to list the contents of the directory
    api_url = f"https://api.github.com/repos/{repo_name}/contents/{dir_path}?ref={branch_name}"

    # Send a request to the GitHub API
    response = requests.get(api_url)
    if response.status_code != 200:
        print(f"Failed to fetch contents from {api_url}. Status code: {response.status_code}")
        return

    files = response.json()
    png_count = 0  # Counter for PNG files

    # Iterate through each file in the directory
    for file in files:
        # Check if the file is a PNG
        if file['name'].endswith('.png'):
            file_url = file['download_url']
            file_name = file['name']
            file_path = os.path.join(output_dir, file_name)

            # Download and save the PNG file
            file_response = requests.get(file_url)
            if file_response.status_code == 200:
                with open(file_path, 'wb') as f:
                    f.write(file_response.content)
                png_count += 1  # Increment the PNG counter
            else:
                print(f"Failed to download {file_name}")

    print(f"Downloaded {png_count} PNG files to {output_dir}")

# Example usage
github_url = "https://github.com/Jef-H/supervised_fields/tree/develop/crop_field_images"
output_directory = "downloaded_png_files/"

download_png_files_from_github(github_url, output_directory)


Cleaned and prepared output directory: downloaded_png_files/
Downloaded 125 PNG files to downloaded_png_files/


In [3]:
def cut_images_to_fragments(input_dir, output_dir, fragment_size=(64, 64)):
    """
    Cuts all PNG images in the input directory into fragments of the given size
    and saves them to the output directory. Excess parts of the images are discarded.

    Args:
        input_dir (str): Directory containing PNG images.
        output_dir (str): Directory to save the image fragments.
        fragment_size (tuple): Size of each fragment (width, height).
    """
    start_time = time.time()  # Start the timer

    # Clear the output directory before writing new fragments
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove the existing directory and its contents
    os.makedirs(output_dir, exist_ok=True)  # Create a fresh directory
    print(f"Ensured output directory is cleared and exists: {output_dir}")

    total_fragment_count = 0  # Counter for all fragments across all images

    # Iterate through each file in the input directory
    for filename in os.listdir(input_dir):
        if filename.endswith('.png'):
            file_path = os.path.join(input_dir, filename)
            with Image.open(file_path) as img:
                img_width, img_height = img.size

                # Calculate the number of fragments in each dimension
                num_fragments_x = img_width // fragment_size[0]
                num_fragments_y = img_height // fragment_size[1]

                total_fragments = 0

                # Generate and save each fragment
                for i in range(num_fragments_x):
                    for j in range(num_fragments_y):
                        left = i * fragment_size[0]
                        upper = j * fragment_size[1]
                        right = left + fragment_size[0]
                        lower = upper + fragment_size[1]

                        fragment = img.crop((left, upper, right, lower))

                        fragment_filename = f"{os.path.splitext(filename)[0]}_{i}_{j}.png"
                        fragment_path = os.path.join(output_dir, fragment_filename)
                        fragment.save(fragment_path)

                        total_fragments += 1
                        total_fragment_count += 1

    print(f"Total fragments created for all images: {total_fragment_count}")

    # End the timer and print the elapsed time
    elapsed_time = time.time() - start_time
    print(f"Processing completed in {elapsed_time:.2f} seconds.")

# Example usage
github_url = "https://github.com/Jef-H/supervised_fields/tree/develop/crop_field_images"
output_directory = "crop_fragments/"

# Create temporary directory outside the 'with' context to avoid deletion
temp_dir = tempfile.mkdtemp()
print(f"Temporary directory created: {temp_dir}")
input_directory = "downloaded_png_files/"

cut_images_to_fragments(input_directory, output_directory)


Temporary directory created: /tmp/tmp5om25c5i
Ensured output directory is cleared and exists: crop_fragments/
Total fragments created for all images: 5343
Processing completed in 9.46 seconds.


In [4]:
# URL of the ZIP file on GitHub
zip_url = 'https://github.com/Jef-H/supervised_fields/blob/develop/Sat_img_class.zip?raw=true'

# Define the directory to save the extracted files
extract_dir = 'satellite_images'

# Step 1: Download the ZIP file
response = requests.get(zip_url)
zip_file_path = 'Sat_img_class.zip'

# Save the file
with open(zip_file_path, 'wb') as f:
    f.write(response.content)

# Step 2: Extract the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f'ZIP file extracted to {extract_dir}')

# Step 3: Optional - Display the extracted files
extracted_files = os.listdir(extract_dir)
print(f'Files extracted: {extracted_files}')

# You can also process the images, e.g., using PIL or other libraries


ZIP file extracted to satellite_images
Files extracted: ['data']


In [5]:
# Set dataset paths
field_images_path = "crop_fragments/"  # Path for field images
not_field_images_paths = [
    "satellite_images/data/cloudy",
    "satellite_images/data/desert",
    "satellite_images/data/green_area",
    "satellite_images/data/water"
]

# Parameters
image_size = (64, 64)  # Image dimensions for resizing

# Helper function to load images
def load_images_from_directory(directory, label):
    """Loads images from a directory and assigns a label."""
    images = []
    labels = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        try:
            img = tf.keras.utils.load_img(filepath, target_size=image_size)
            img_array = tf.keras.utils.img_to_array(img) / 255.0  # Normalize
            images.append(img_array)
            labels.append(label)
        except Exception as e:
            print(f"Error loading image {filename}: {e}")
    return np.array(images), np.array(labels)

# Load field data
field_images, field_labels = load_images_from_directory(field_images_path, label="field")

# Load not-field data from multiple directories
not_field_images = []
not_field_labels = []
for path in not_field_images_paths:
    images, labels = load_images_from_directory(path, label="not_field")
    not_field_images.append(images)
    not_field_labels.append(labels)

try:
    not_field_images = np.concatenate(not_field_images, axis=0)
    not_field_labels = np.concatenate(not_field_labels, axis=0)
except ValueError:
    print("Error: No images loaded for the 'not_field' category.")

# Combine data
try:
    X = np.concatenate([field_images, not_field_images], axis=0)
    y = np.concatenate([field_labels, not_field_labels], axis=0)
except ValueError:
    print("Error: Ensure both 'field' and 'not_field' data are loaded correctly.")
    X, y = None, None

if X is not None and y is not None:
    # Encode labels
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)  # Convert "field"/"not_field" to 0/1

    # Split data
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Data augmentation for CNNs
    data_gen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True
    )

    data_gen.fit(X_train)

    print("Data preparation complete:")
    print(f"Training data: {X_train.shape}, {y_train.shape}")
    print(f"Validation data: {X_val.shape}, {y_val.shape}")
    print(f"Test data: {X_test.shape}, {y_test.shape}")
else:
    print("Data preparation failed. Check the dataset and paths.")



Data preparation complete:
Training data: (7681, 64, 64, 3), (7681,)
Validation data: (1646, 64, 64, 3), (1646,)
Test data: (1647, 64, 64, 3), (1647,)


In [6]:

# Build CNN model
model = Sequential([
    Input(shape=(64, 64, 3)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    BatchNormalization(),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    BatchNormalization(),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    BatchNormalization(),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 31, 31, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 14, 14, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 6, 6, 128)           │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         589,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 684,225 (2.61 MB)

 Trainable params: 683,777 (2.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [7]:
# Define the CNN model
model = Sequential([
    # First convolutional block
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Second convolutional block
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Third convolutional block
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    GlobalAveragePooling2D(),

    # Dense layers
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Print model summary
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 62, 62, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 29, 29, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 12, 12, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 102,465 (400.25 KB)

 Trainable params: 102,017 (398.50 KB)

 Non-trainable params: 448 (1.75 KB)

### Comparing the Two CNNs

Here’s a breakdown of the differences between the original CNN and the smaller one:

---

#### 1. Model Complexity (Total Parameters)
- **Original CNN**: 684,225 total parameters (2.61 MB).  
- **Smaller CNN**: 102,465 total parameters (400.25 KB).  

The smaller CNN has about 85% fewer parameters, making it way lighter and faster to train, which is great if you're looking for efficiency.

---

#### 2. Output Reduction
- **Original CNN**:  
  - Final output from the convolutional layers: `(6, 6, 128)` → Flattened to `4608`.  
  - Fully connected layers include a dense layer with 128 neurons followed by the output layer.

- **Smaller CNN**:  
  - Final output from the convolutional layers: `(12, 12, 128)` → Reduced with `GlobalAveragePooling2D` to `(128)`.  
  - Fully connected layers use a smaller dense layer with just 64 neurons before the output.

**Key Difference**:  
The smaller CNN relies on `GlobalAveragePooling2D` instead of flattening, which slashes the parameter count while still keeping the important features.

---

#### 3. Spatial Reduction
- **Original CNN**:  
  - Reduces spatial dimensions to `(6, 6, 128)` through pooling. This could lead to a loss of details.  

- **Smaller CNN**:  
  - Keeps the spatial dimensions larger for longer (`(12, 12, 128)`), preserving more spatial information before pooling.

---

### Recommendations
If your dataset is smaller or you're tight on training time, the smaller CNN is the better option. But if you’ve got a large dataset and more resources, you could go with the original CNN—just tweak it to avoid overfitting or extra complexity.


### Support Vector Machine (SVM) Implementation
The following code is going to make a Support Vector Machine (SVM) for binary classification.

The SVM will:
1. Use the `rbf` kernel (Gaussian kernel) for non-linear classification.
2. Employ hyperparameter tuning using `GridSearchCV` to find the best `C` and `gamma` values.
3. Evaluate the model using metrics like accuracy and a confusion matrix.

In [8]:
print(X_train.shape)
print(X_test.shape)

(7681, 64, 64, 3)
(1647, 64, 64, 3)


In [9]:
# Flatten the images into 2D arrays to work with the standard scaler
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Now scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

In [ ]:

# Flatten the dataset (if not already flattened)
X_flat = X.reshape(X.shape[0], -1)

# Fit PCA to the data without specifying the number of components
pca = PCA()
pca.fit(X_flat)

# Plot cumulative explained variance
cumulative_variance = pca.explained_variance_ratio_.cumsum()
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
plt.axhline(y=0.95, color='r', linestyle='--')  # 95% threshold
plt.title('Cumulative Explained Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.show()

# Find the minimum number of components for 95% variance
n_components_95 = next(i for i, total in enumerate(cumulative_variance) if total >= 0.95) + 1
print(f"Number of components for 95% variance: {n_components_95}")

In [ ]:
# Start timing the entire script
script_start = time.time()

# Flatten the images into 2D arrays before splitting
start = time.time()
X_flat = X.reshape(X.shape[0], -1)
end = time.time()
print(f"Flattening images took {end - start:.2f} seconds.")

# Apply PCA for dimensionality reduction
start = time.time()
pca = PCA(n_components=200)  # Adjust n_components as needed
X_flat_pca = pca.fit_transform(X_flat)
end = time.time()
print(f"PCA transformation took {end - start:.2f} seconds.")

# Split your data into training and testing sets
start = time.time()
X_train_flat, X_test_flat, y_train, y_test = train_test_split(X_flat_pca, y, test_size=0.3, random_state=42)
end = time.time()
print(f"Data splitting took {end - start:.2f} seconds.")

# Scale the data for SVM
start = time.time()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)
end = time.time()
print(f"Data scaling took {end - start:.2f} seconds.")

# Define the SVM model
svm = SVC()

# Define a parameter grid for hyperparameter tuning
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

# Perform GridSearchCV for hyperparameter tuning
start = time.time()
grid = GridSearchCV(svm, param_grid, refit=True, cv=5, scoring='accuracy')
grid.fit(X_train_scaled, y_train)
end = time.time()
print(f"GridSearchCV took {end - start:.2f} seconds.")

# Print the best parameters
print("Best parameters:", grid.best_params_)

# Evaluate the best model on the test set
start = time.time()
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)
end = time.time()
print(f"Model evaluation and prediction took {end - start:.2f} seconds.")

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# End timing the entire script
script_end = time.time()
print(f"The entire script took {script_end - script_start:.2f} seconds.")

Overall, the model performed very well. This suggests that the SVM with the chosen hyperparameters is a strong model for this binary classification task.

# Reflection Questions

Answer the following questions about your supervised learning classification project. Write 1-3 lines per answer in separate Markdown cells.

---

### 1. Which method did you like the most?
I like the CNN more in this context because:

    It leverages the spatial structure of image data.
    Although it uses more compute it avoids the information loss caused by PCA and flattening in the SVM pipeline.
    CNNs tend to generalize better for larger and more complex datasets, especially as the task scales.

---

### 2. Which method did you like the least?
PCA + SVM because I hate that I had to volunteer to leave data/knowledge on the table by reducing the number of componets with PCA, but I had to use PCA because SVM took so long.


---

### 3. How did you score these supervised models?  
For CNN we used accuracy as the performance metric. For SVM, the notebook performs GridSearchCV to find the best hyperparameters using cross-validation. The scoring metric used during the search is accuracy.


---

### 4. Did the output align with your geologic understanding?  
N/A

---

### 5. Did you hyperparameter tune? Why or why not?  

SVM used hyperparameter tuning, CNN did not use hyperparameter tuning. The hyperparameter tuning on the SVM helps find the C, gamma, and kernel that work for my dataset. This leads to more accurate classifications and better generalizations.

---

### 6. How did you split your data? And why does that make sense for this dataset?  
Train_test_split method was used to split the data. It shuffles the data and then randomply selects datapoints for the training and testing datasets. Random splitting is appropriate for my dataset because there's not a time factor involved in the classification question I'm trying to solve.

However, the cornfield data set could be skewed because they were gathered with 125 screen shots, and then those screen shots were broken down into field fragments. And so I believe it's likely the model has seen images from that same field in the trainingset. I should take some new screenshots it hasn't seen and verify. Or consider splitting the data before creating the fragments to ensure the model could generalize effectively.

---

### 7. What did you want to learn more about?  
I think future tinkering, may include adding additonal classes, so instead of field or not a field I'd like to combine all the catagories listed, so given a satellite image, return a grid with the classsifications of each 64x64 piece of land based on the results of this dataset.


---

### 8. Did you pre-process your data?  

Data splitting was used to break the data into train and test baches, then for the CNN, images were reshaped so they were consistent. Lastly, we normalized pixel values to get them from 0-255 to a value between 0 and 1

For the SVM the Standard scaler was used on SVM


# Conclusion

This notebook explored two supervised learning methods, 2 CNNs and 1 SVM, for classifying images as fields or not fields. The CNN leveraged its ability to process spatial data, resulting in better generalization and avoiding the need for dimensionality reduction. SVM, while effective, required PCA to reduce dimensionality, leading to some information being discarded. Both methods used accuracy as the primary performance metric.

Data was split using train_test_split, but the dataset's fragmentation method raised concerns about potential contaminaton between training and testing images.

A future suggestion would be to re-split the dataset before fragmenting the screenshots to ensure the model's generalization capabilities are robust. One last thought, is exploring hyperparameter tuning for the CNN might provide additional performance.